In [1]:
import os

In [2]:
%pwd

'/Users/Apple/ZAINUL_AI/mlflow/ml_1/research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/Users/Apple/ZAINUL_AI/mlflow/ml_1'

In [69]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelTrainerConfig:
    root_dir: Path
    train_data_path: Path
    test_data_path: Path
    model_name: str
    alpha: float
    l1_ratio: float
    target_column: str

In [70]:
from src.datascience.constants import *
from src.datascience.utils.common import read_yaml, create_directories


In [71]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_trainer_config(self) -> ModelTrainerConfig:

        config = self.config.model_trainer
        params = self.params.ElasticNet

        # TARGET_COLUMN ek string hai
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir=Path(config.root_dir),
            train_data_path=Path(config.train_data_path),
            test_data_path=Path(config.test_data_path),
            model_name=config.model_name,
            alpha=params.alpha,
            l1_ratio=params.l1_ratio,
            target_column=schema
        )

        return model_trainer_config

In [72]:
import os
import pandas as pd
import joblib

from src.datascience import logger
from sklearn.linear_model import ElasticNet

In [ ]:



class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train(self):

        # Load training and testing datasets
        train_data = pd.read_csv(self.config.train_data_path)
        test_data = pd.read_csv(self.config.test_data_path)

        logger.info("Training and testing data loaded")

        # Get target column from configuration
        target_column = self.config.target_column

        # Check whether target column exists
        if target_column not in train_data.columns:
            raise ValueError(
                f"Target column '{target_column}' not found in training data"
            )

        if target_column not in test_data.columns:
            raise ValueError(
                f"Target column '{target_column}' not found in testing data"
            )

        # Split features and target
        train_x = train_data.drop(columns=[target_column])
        test_x = test_data.drop(columns=[target_column])

        train_y = train_data[target_column]
        test_y = test_data[target_column]

        logger.info("Data split into features and target")

        # Initialize ElasticNet model
        model = ElasticNet(
            alpha=self.config.alpha,
            l1_ratio=self.config.l1_ratio,
            random_state=42
        )

        # Train model
        model.fit(train_x, train_y)

        logger.info("Model training completed")

        # Create output directory
        os.makedirs(self.config.root_dir, exist_ok=True)

        # Save trained model
        model_path = os.path.join(
            self.config.root_dir,
            self.config.model_name
        )

        joblib.dump(model, model_path)

        logger.info(f"Model saved successfully at: {model_path}")

        # Optional: return model and test data for evaluation
        return model, test_x, test_y

In [78]:
try:
    config = ConfigurationManager()

    model_trainer_config = config.get_model_trainer_config()

    model_trainer = ModelTrainer(
        config=model_trainer_config
    )

    model, test_x, test_y = model_trainer.train()

    print("Model training successful!")

except Exception as e:
    logger.exception("Error occurred during model training")
    raise

[2026-10-23 17:05:05,027: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-10-23 17:05:05,029: INFO: common: yaml file: params.yaml loaded successfully]
[2026-10-23 17:05:05,031: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-10-23 17:05:05,032: INFO: common: created directory at: artifacts]
[2026-10-23 17:05:05,032: INFO: common: created directory at: artifacts/model_trainer]
[2026-10-23 17:05:05,039: INFO: 2852111771: Model saved successfully at artifacts/model_trainer/model.joblib]
Model training successful!
